# transformer-residual-lab · Kaggle 低位宽量化验证台

**版本** v0.1（首版）· **目标 checkpoint** `Qwen/Qwen3-8B`（8.19 B 参数，36 层，hidden 4096，GQA 32/8，head_dim 128，vocab 151936）

这个 Notebook 只做一件事：在**单张 T4 16 GB** 上，用**同一份 base GGUF + 同一份 imatrix**产出 `IQ2_XS` 与 `IQ4_XS`，然后比较**实际文件字节**、**配对 NLL**、以及**有限的配对质量指标**。

它**不会**：
- 自动下载任何大模型（`ALLOW_NETWORK = False`）
- 自动 `pip install`（全部依赖 Kaggle 镜像已有，llama.cpp 从源码编译）
- 自动租 GPU、自动跑实验
- 上传任何私有项目文件

**首版默认只开 `S0_ENV` / `S1_SMOKE` / `S9_REPORT`。** 其余阶段全部默认 `False`，需要你在下面的配置区逐个打开。

---

## 为什么必须这么小心

| 事实 | 后果 |
|---|---|
| `llama-quant.cpp:882` 的 `tensor_requires_imatrix` 把 `GGML_TYPE_IQ2_XS` 列进 `switch` 并 `return true`；`:1324` 在缺 imatrix 时 `throw std::runtime_error("The result will be garbage, so bailing out")` | `llama-quantize ... IQ2_XS` 在无 imatrix 时**直接退出、不产出文件**。所以 S4 强制要求 imatrix，缺失即报错，不做降级 |
| `IQ4_XS` **不**要求 imatrix，`IQ2_XS` 要求 | 只给 Q2 配 imatrix 就是给低比特开小灶。**两者共用同一份 `imatrix.gguf`** |
| imatrix 会改变 `llama_tensor_get_type` 的分层决策（`llama-quant.cpp:688` 对前 `n_layer/8` 层按 `has_imatrix` 分支） | 两个文件的实际张量类型组合不同，**字节必须从文件读，不能用名义 bpw 算** |
| T4 是 Turing（CC 7.5），**没有原生 bf16** | 高精度锚点在 T4 上只能用 **f16**，不是 bf16。这是一处必须记录的偏差 |
| 8B 权重：f16 = 16.38 GB = **15.25 GiB**；T4 16 GB 卡可用约 **15.0 GiB** | **单张 T4 装不下完整 8B 高精度权重**（还差 KV 与 CUDA context）。S8 只保留接口与可行性判定，默认拒绝执行 |
| 第三方现成量化 GGUF 的校准集、imatrix、量化脚本都未知 | 它**只能用于探索性测试**，不能与自建量化放进同一张公平比较表。S9 有断言强制隔离 |


## 阶段总览

| 阶段 | 默认 | 依赖 | 预算 | 产出 |
|---|---|---|---|---|
| `S0_ENV` | **开** | 无 | 120 s | GPU/CPU/RAM/磁盘/工具链/`/kaggle/input` 清单 |
| `S1_SMOKE` | **开** | 无 | 240 s | 合成 GGUF 往返自检、fp16/bf16 矩阵乘、拷贝带宽、磁盘写速 |
| `S2_BUILD` | 关 | llama.cpp 源码 | 2400 s | `llama-imatrix` / `llama-quantize` / `llama-perplexity` / `llama-cli` |
| `S3_IMATRIX` | 关 | S2 + base GGUF + 校准语料 | 3600 s | `imatrix.gguf` |
| `S4_QUANT` | 关 | S2 + S3 + base GGUF | 1800 s | `IQ2_XS` / `IQ4_XS` 两个 GGUF |
| `S5_BYTES` | 关 | S4（或任何 GGUF） | 300 s | 逐张量类型的真实字节与 bpw |
| `S6_NLL` | 关 | S2 + S5 + 评测语料 | 5400 s | 配对 NLL、配对 bootstrap CI |
| `S7_QUALITY` | 关 | S2 + S5 | 1800 s | 贪心续写一致性、重复度 |
| `S8_BF16_ANCHOR` | 关 | 高精度 GGUF + 足够显存 | 3600 s | 锚点 NLL，或一条明确的 skip 记录 |
| `SX_EXPLORE` | 关 | 第三方 GGUF | 1800 s | 字节 + NLL，**标记 `provenance=third_party`** |
| `S9_REPORT` | **开** | 无 | 120 s | `report.md` + `results.json` |

每个阶段都受 **墙钟 / 显存 / 磁盘** 三重上限约束；状态写入 `trlab/ckpt/state.json`，重跑时已完成的阶段自动跳过（`FORCE_RERUN=True` 可强制重跑）。

---

## 你需要先准备的东西

在 Kaggle 里**手动**挂载为 Dataset（只读挂在 `/kaggle/input/<slug>/`）：

1. **base GGUF**：`Qwen3-8B` 的 f16 或 Q8_0。f16 是 15.25 GiB，Q8_0 约 8.1 GiB（= 8.19e9 × 8.5/8）。只读挂载不占 `/kaggle/working` 的 20 GB 配额。
   - 为什么可能需要 Q8_0：imatrix 与 quantize 都要加载整模型。f16 在 T4 上放不进显存，只能 CPU 跑（很慢）。Q8_0 能进显存。
   - **用哪个 base 必须写进 `CFG["BASE_KIND"]`**，它会进 provenance。
2. **校准语料** `calib.txt`（imatrix 用）
3. **评测语料** `eval.txt`（NLL 用，**必须与校准集不同源**）
4. **llama.cpp 源码**（可选，若不想在 Notebook 里 `git clone`）

第 2、3 项建议都用 wikitext-2-raw 的**不同 split**，或者干脆不同数据集。S3 会做内容级重叠断言。


In [ ]:
# =====================================================================
# 唯一需要改动的区域。
# 默认只开 S0_ENV / S1_SMOKE / S9_REPORT，其余全部关闭。
# 关闭的阶段不会执行、不会联网、不会下载任何东西。
# =====================================================================
from pathlib import Path

STAGES = {
    "S0_ENV":         True,    # 环境 / GPU / 磁盘检查（秒级）
    "S1_SMOKE":       True,    # 短时 smoke：合成 GGUF 往返 + 显存/磁盘微基准
    "S2_BUILD":       False,   # 从源码构建 llama.cpp（CPU 或 CUDA）
    "S3_IMATRIX":     False,   # 生成 imatrix.gguf
    "S4_QUANT":       False,   # 产出 IQ2_XS / IQ4_XS（共用同一份 imatrix）
    "S5_BYTES":       False,   # 读文件算真实字节与 bpw
    "S6_NLL":         False,   # 配对 NLL + bootstrap CI
    "S7_QUALITY":     False,   # 有限配对质量检查
    "S8_BF16_ANCHOR": False,   # 预留接口：高精度锚点（T4 上会主动拒绝）
    "S9_REPORT":      True,    # 汇总
    "SX_EXPLORE":     False,   # 第三方现成量化，仅探索
}

CFG = {
    # ---------- 模型 ----------
    "MODEL_ID":    "Qwen/Qwen3-8B",
    "PARAM_COUNT": 8.19e9,
    "N_LAYERS":    36,
    "HIDDEN":      4096,
    "N_KV_HEADS":  8,
    "HEAD_DIM":    128,

    # ---------- 文件路径（全部要求手动挂载到 /kaggle/input） ----------
    "BASE_GGUF":    "",   # 例 "/kaggle/input/qwen3-8b-gguf/qwen3-8b-f16.gguf"
    "BASE_KIND":    "",   # 必须与文件一致："f16" / "bf16" / "q8_0" / "q4_k_m" ...
    "IMATRIX_GGUF": "",   # 留空则由 S3 生成到 working
    "CALIB_TXT":    "",   # imatrix 校准语料
    "EVAL_TXT":     "",   # NLL 评测语料（必须与校准集不同源）
    "EVAL_SOURCE":  "",   # 例 "wikitext-2-raw/test"，写进 provenance
    "CALIB_SOURCE": "",   # 例 "wikitext-2-raw/train"
    "EXPLORE_GGUF": "",   # 第三方现成量化，仅探索

    # ---------- llama.cpp ----------
    "LLAMA_SRC":  "",      # 源码目录；留空则尝试 /kaggle/input/llama.cpp
    "LLAMA_BIN":  "",      # 已有构建的 bin 目录；留空则用 <LLAMA_SRC>/build/bin
    "BUILD_CUDA": True,    # False 则只构建 CPU 版（也能跑 imatrix / quantize）
    "BUILD_JOBS": 4,

    # ---------- 预算（硬上限，超了主动抛异常而不是被 OOM kill） ----------
    "MAX_WALL_SEC":      8 * 3600,
    "MAX_VRAM_GB":       14.0,
    "MIN_FREE_DISK_GB":  3.0,
    "STAGE_BUDGET_SEC": {
        "S0_ENV": 120, "S1_SMOKE": 240, "S2_BUILD": 2400,
        "S3_IMATRIX": 3600, "S4_QUANT": 1800, "S5_BYTES": 300,
        "S6_NLL": 5400, "S7_QUALITY": 1800, "S8_BF16_ANCHOR": 3600,
        "S9_REPORT": 120, "SX_EXPLORE": 1800,
    },

    # ---------- 实验规模（首版刻意保守） ----------
    "IMATRIX_CTX":           512,
    "IMATRIX_CHUNKS":        32,
    "NLL_CTX":               2048,
    "NLL_CHUNKS_PER_GROUP":  4,
    "NLL_GROUPS":            8,
    "QUALITY_PROMPTS":       8,
    "QUALITY_N_PREDICT":     32,

    # ---------- 开关 ----------
    "FORCE_RERUN":      False,   # True 则忽略断点，全部重跑
    "ALLOW_NETWORK":    False,   # 保持 False。True 才允许 git clone llama.cpp
    "ALLOW_CPU_ANCHOR": False,   # True 才允许在显存放不下时用 CPU 跑高精度锚点
    "HASH_GGUF":        True,    # 对产物算 sha256（2.4 GB 约 10 s）
    "BOOTSTRAP_N":      10000,
    "SEED":             1234,
}

# 这条永远不允许改成 True。S9 会断言。
THIRD_PARTY_IN_COMPARISON = False

# ---- 派生路径 ----
ON_KAGGLE = Path("/kaggle/working").is_dir()
ROOT = Path("/kaggle/working") if ON_KAGGLE else Path.cwd() / "_trlab_run"
WORK = ROOT / "trlab"
OUT  = WORK / "out"
LOGS = WORK / "logs"
CKPT = WORK / "ckpt"
MDLS = WORK / "models"
IN   = Path("/kaggle/input")

print(f"运行位置      : {'Kaggle' if ON_KAGGLE else '本地/其他'}")
print(f"工作目录      : {WORK}")
print(f"开启的阶段    : {[k for k, v in STAGES.items() if v]}")
print(f"网络下载      : {'允许' if CFG['ALLOW_NETWORK'] else '禁止'}")
print(f"第三方量化入表: {'允许（不应发生）' if THIRD_PARTY_IN_COMPARISON else '禁止'}")


In [ ]:
import json, os, re, sys, time, math, hashlib, platform, shutil, struct, subprocess, traceback
from datetime import datetime, timezone

for _d in (WORK, OUT, LOGS, CKPT, MDLS):
    _d.mkdir(parents=True, exist_ok=True)

T0 = time.time()
STATE_PATH = CKPT / "state.json"


def log(msg):
    print(f"[{time.strftime('%H:%M:%S')} +{time.time() - T0:7.1f}s] {msg}", flush=True)


def write_json(path, obj):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=1, ensure_ascii=False, default=str), encoding="utf-8")
    tmp.replace(path)
    return path


def load_state():
    if STATE_PATH.exists():
        try:
            return json.loads(STATE_PATH.read_text(encoding="utf-8"))
        except Exception:
            log("state.json 损坏，重新开始")
    return {}


STATE = load_state()


def save_state():
    write_json(STATE_PATH, STATE)


class BudgetExceeded(RuntimeError):
    """墙钟 / 显存 / 磁盘超限。故意抛异常，而不是等 OOM killer。"""


class GuardError(RuntimeError):
    """前置条件不满足（路径缺失、二进制缺失、开关未打开）。"""


# ---------------- 资源观测 ----------------

def vram_gb():
    try:
        import torch
        if not torch.cuda.is_available():
            return None
        free, total = torch.cuda.mem_get_info()
        return {"free": free / 2**30, "total": total / 2**30, "used": (total - free) / 2**30}
    except Exception:
        return None


def disk_gb(p):
    p = Path(p)
    p.mkdir(parents=True, exist_ok=True)
    u = shutil.disk_usage(p)
    return {"total": u.total / 2**30, "used": u.used / 2**30, "free": u.free / 2**30}


def guard(tag):
    el = time.time() - T0
    if el > CFG["MAX_WALL_SEC"]:
        raise BudgetExceeded(
            f"{tag}: 累计墙钟 {el/3600:.2f} h 超过上限 {CFG['MAX_WALL_SEC']/3600:.2f} h")
    v = vram_gb()
    if v and v["used"] > CFG["MAX_VRAM_GB"]:
        raise BudgetExceeded(
            f"{tag}: 显存占用 {v['used']:.2f} GB 超过上限 {CFG['MAX_VRAM_GB']} GB")
    d = disk_gb(WORK)
    if d["free"] < CFG["MIN_FREE_DISK_GB"]:
        raise BudgetExceeded(
            f"{tag}: 磁盘剩余 {d['free']:.2f} GB 低于下限 {CFG['MIN_FREE_DISK_GB']} GB")


def check_deadline(deadline, tag):
    if time.time() > deadline:
        raise BudgetExceeded(f"{tag}: 阶段预算已用尽")


# ---------------- 阶段执行器 ----------------

def run_stage(name, fn):
    """阶段开关 + 断点恢复 + 预算 + JSON 落盘的统一入口。"""
    if not STAGES.get(name, False):
        log(f"[{name}] SKIP —— STAGES['{name}'] 为 False")
        return None

    prev = STATE.get(name) or {}
    if prev.get("status") == "ok" and not CFG["FORCE_RERUN"]:
        log(f"[{name}] RESUME —— 已完成，跳过。产物 {prev.get('result_path', '')}")
        return prev.get("result")

    budget = CFG["STAGE_BUDGET_SEC"].get(name, 600)
    log(f"[{name}] START（预算 {budget} s）")
    t = time.time()
    rec = {"stage": name, "started": datetime.now(timezone.utc).isoformat(timespec="seconds")}
    try:
        guard(name)
        result = fn(t + budget)
        rec.update(status="ok", elapsed_sec=round(time.time() - t, 2),
                   result=result, result_path=str(OUT / f"{name}.json"))
        write_json(OUT / f"{name}.json", {"stage": name, "meta": rec, "result": result})
        log(f"[{name}] OK  {rec['elapsed_sec']} s  →  {rec['result_path']}")
    except Exception as e:
        rec.update(status="error", elapsed_sec=round(time.time() - t, 2),
                   error=f"{type(e).__name__}: {e}",
                   traceback=traceback.format_exc()[-2000:])
        write_json(OUT / f"{name}.json", {"stage": name, "meta": rec, "result": None})
        log(f"[{name}] FAIL  {rec['error']}")

    STATE[name] = rec
    save_state()
    return rec.get("result")


# ---------------- 外部命令 ----------------

def run_cmd(cmd, log_path, deadline, cwd=None, env=None, check=True):
    """跑外部命令，输出落日志，超预算就 kill。"""
    cmd = [str(c) for c in cmd]
    log(f"  $ {' '.join(cmd)}")
    t = time.time()
    with open(log_path, "wb") as lf:
        p = subprocess.Popen(cmd, stdout=lf, stderr=subprocess.STDOUT, cwd=cwd, env=env)
        while p.poll() is None:
            if time.time() > deadline:
                p.kill()
                raise BudgetExceeded(f"命令超预算被杀: {cmd[0]}")
            time.sleep(0.5)
    rc = p.returncode
    tail = Path(log_path).read_text(errors="replace")[-1800:]
    log(f"  rc={rc}  {time.time() - t:.1f} s  →  {log_path}")
    if check and rc != 0:
        raise RuntimeError(f"命令失败 rc={rc}: {' '.join(cmd)}\n--- tail ---\n{tail}")
    return rc, tail


def need(key):
    """取一个必须存在的路径配置。"""
    v = CFG.get(key, "")
    if not v:
        raise GuardError(f"CFG['{key}'] 未设置")
    p = Path(v)
    if not p.exists():
        raise GuardError(f"CFG['{key}'] 指向的路径不存在: {p}")
    return p


def llama_bin(name):
    """按 CFG -> 源码树 -> PATH 的顺序找 llama.cpp 可执行文件。"""
    if CFG["LLAMA_BIN"]:
        p = Path(CFG["LLAMA_BIN"]) / name
        if p.exists():
            return p
    src = CFG["LLAMA_SRC"] or "/kaggle/input/llama.cpp"
    for cand in (Path(src) / "build" / "bin" / name, Path(src) / "build" / name):
        if cand.exists():
            return cand
    w = shutil.which(name)
    if w:
        return Path(w)
    raise GuardError(f"找不到 {name}。先跑 S2_BUILD，或设置 CFG['LLAMA_BIN']")


def sha256_of(path, deadline=None, chunk=8 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
            if deadline and time.time() > deadline:
                return "aborted:deadline"
    return h.hexdigest()


def paired_bootstrap(diffs, n=None, seed=None):
    """配对差值的 bootstrap 均值 CI。"""
    import random
    n = n or CFG["BOOTSTRAP_N"]
    seed = CFG["SEED"] if seed is None else seed
    m = len(diffs)
    if m == 0:
        return {"n": 0, "mean": None, "ci95": None}
    mean = sum(diffs) / m
    if m < 2:
        return {"n": m, "mean": mean, "ci95": None, "note": "只有 1 对，无法给 CI"}
    rng = random.Random(seed)
    means = []
    for _ in range(n):
        s = 0.0
        for _ in range(m):
            s += diffs[rng.randrange(m)]
        means.append(s / m)
    means.sort()
    return {"n": m, "mean": mean,
            "ci95": [means[int(0.025 * n)], means[int(0.975 * n)]],
            "boot": n}


log("运行时已就绪")
log(f"断点状态: {list(STATE.keys()) if STATE else '空'}")


## S0 · 环境 / GPU / 磁盘检查

这一步不加载任何模型，只回答「这台机器能干什么」。

In [ ]:
def stage_s0_env(deadline):
    info = {}
    info["where"] = "kaggle" if ON_KAGGLE else "local"
    info["utc"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
    info["python"] = sys.version.split()[0]
    info["platform"] = platform.platform()
    info["cpu_count"] = os.cpu_count()

    # ---- RAM ----
    ram = None
    try:
        import psutil
        vm = psutil.virtual_memory()
        ram = {"total_gb": round(vm.total / 2**30, 2), "available_gb": round(vm.available / 2**30, 2)}
    except Exception:
        try:
            pages = os.sysconf("SC_PHYS_PAGES")
            psize = os.sysconf("SC_PAGE_SIZE")
            ram = {"total_gb": round(pages * psize / 2**30, 2), "available_gb": None}
        except Exception as e:
            ram = {"error": repr(e)}
    info["ram"] = ram

    # ---- GPU ----
    gpus = []
    try:
        import torch
        info["torch"] = torch.__version__
        info["torch_cuda"] = torch.version.cuda
        info["cuda_available"] = bool(torch.cuda.is_available())
        for i in range(torch.cuda.device_count()):
            p = torch.cuda.get_device_properties(i)
            free, total = torch.cuda.mem_get_info(i)
            gpus.append({
                "idx": i, "name": p.name, "cc": f"{p.major}.{p.minor}",
                "total_gb": round(total / 2**30, 2),
                "free_gb": round(free / 2**30, 2),
                "sm_count": p.multi_processor_count,
                # CC < 8.0 没有原生 bf16（T4 = 7.5）
                "native_bf16": p.major >= 8,
            })
    except Exception as e:
        info["torch_error"] = repr(e)
    info["gpus"] = gpus

    if gpus:
        # CUDA context + cuBLAS workspace 大约吃掉 0.5-1.0 GB
        info["vram_usable_gb_est"] = round(max(g["total_gb"] for g in gpus) - 1.0, 2)
    else:
        info["vram_usable_gb_est"] = 0.0

    try:
        info["nvidia_smi"] = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
             "--format=csv,noheader"],
            capture_output=True, text=True, timeout=30).stdout.strip()
    except Exception as e:
        info["nvidia_smi"] = f"(unavailable: {e})"

    # ---- 磁盘 ----
    disks = {}
    for p in ["/kaggle/working", "/kaggle/temp", "/kaggle/input", str(WORK)]:
        try:
            if Path(p).exists():
                disks[p] = {k: round(v, 2) for k, v in disk_gb(p).items()}
        except Exception as e:
            disks[p] = {"error": repr(e)}
    info["disks_gb"] = disks

    # ---- 工具链 ----
    tools = {}
    for b in ["cmake", "g++", "gcc", "nvcc", "git", "make",
              "llama-imatrix", "llama-quantize", "llama-perplexity", "llama-cli"]:
        tools[b] = shutil.which(b)
    info["tools"] = tools

    # ---- Python 包 ----
    pkgs = {}
    for name in ["torch", "numpy", "psutil", "gguf", "transformers", "datasets"]:
        try:
            import importlib.metadata as md
            pkgs[name] = md.version(name)
        except Exception:
            pkgs[name] = None
    info["packages"] = pkgs

    # ---- /kaggle/input 清单（这是你填 CFG 路径的依据）----
    inv = []
    if IN.is_dir():
        for d in sorted(IN.iterdir()):
            if not d.is_dir():
                continue
            n_files, tot, ggufs = 0, 0, []
            try:
                for j, p in enumerate(d.rglob("*")):
                    if j % 200 == 0:
                        check_deadline(deadline, "S0_ENV 扫描 /kaggle/input")
                    if p.is_file():
                        n_files += 1
                        try:
                            sz = p.stat().st_size
                        except OSError:
                            continue
                        tot += sz
                        if p.suffix.lower() == ".gguf":
                            ggufs.append({"path": str(p), "gb": round(sz / 2**30, 3)})
            except Exception as e:
                inv.append({"dataset": d.name, "error": repr(e)})
                continue
            inv.append({"dataset": d.name, "files": n_files,
                        "gb": round(tot / 2**30, 2), "gguf": ggufs[:20],
                        "gguf_count": len(ggufs)})
    info["input_datasets"] = inv

    # ---- 结论性提示 ----
    warn = []
    if not gpus:
        warn.append("没有可用 GPU。S6/S7 的 llama.cpp 会退到 CPU，速度慢几十倍。")
    elif any(not g["native_bf16"] for g in gpus):
        warn.append("GPU 不支持原生 bf16（CC<8.0）。高精度锚点必须用 f16，"
                    "且 bf16 矩阵乘会走模拟路径 —— S1 会实测这个比值。")
    if info["vram_usable_gb_est"] and info["vram_usable_gb_est"] < 16.4:
        warn.append(f"可用显存约 {info['vram_usable_gb_est']:.1f} GB，"
                    "装不下 8B 高精度权重（f16 = 15.25 GiB，再加 KV 与 CUDA context）。S8 会拒绝执行。")
    if not any(x.get("gguf_count") for x in inv if isinstance(x, dict)):
        warn.append("/kaggle/input 下没有找到 .gguf。S3-S7 需要你手动挂载 base GGUF。")
    if tools["cmake"] is None:
        warn.append("没有 cmake，S2_BUILD 无法执行。")
    info["warnings"] = warn

    # ---- 打印 ----
    print("=" * 74)
    print(f"Python {info['python']} | CPU {info['cpu_count']} | RAM {ram}")
    for g in gpus:
        print(f"GPU{g['idx']}: {g['name']}  CC{g['cc']}  {g['total_gb']} GB "
              f"(free {g['free_gb']})  SM={g['sm_count']}  native_bf16={g['native_bf16']}")
    print(f"可用显存估计: {info['vram_usable_gb_est']} GB")
    print("-" * 74)
    for k, v in disks.items():
        print(f"disk {k:20s} total={v.get('total')} GB  free={v.get('free')} GB")
    print("-" * 74)
    print("tools:", {k: (v.split("/")[-1] if v else None) for k, v in tools.items()})
    print("pkgs :", pkgs)
    print("-" * 74)
    if inv:
        print("/kaggle/input:")
        for d in inv:
            if "error" in d:
                print(f"  {d['dataset']}: ERROR {d['error']}")
            else:
                print(f"  {d['dataset']}: {d['files']} files, {d['gb']} GB, "
                      f"{d['gguf_count']} gguf")
                for g in d["gguf"]:
                    print(f"      {g['gb']:8.2f} GB  {g['path']}")
    else:
        print("/kaggle/input 为空")
    print("=" * 74)
    for w in warn:
        print("!! " + w)

    return info


s0 = run_stage("S0_ENV", stage_s0_env)


## 关于 GGUF 字节统计

`S5_BYTES` 用一个**纯 Python 的 GGUF 头解析器**（约 90 行，无第三方依赖）逐张量读出 `type` / `dims` / `offset`，按 ggml 的 block 结构算真实字节：

```
bytes(tensor) = ceil(n_elements / block_size) * type_size
```

关键 block 尺寸（来自 `ggml-common.h` 的结构体实际大小）：

| 类型 | block | bytes | bpw |
|---|---|---|---|
| `F16` | 1 | 2 | 16.0 |
| `BF16` | 1 | 2 | 16.0 |
| `Q4_K` | 256 | 144 | 4.5 |
| `Q6_K` | 256 | 210 | 6.5625 |
| `Q8_0` | 32 | 34 | 8.5 |
| `IQ2_XS` | 256 | 74 | **2.3125** |
| `IQ4_XS` | 256 | 136 | **4.25** |
| `IQ2_XXS` | 256 | 66 | 2.0625 |
| `TQ2_0` | 256 | 66 | 2.0625 |
| `MXFP4` | 32 | 17 | 4.25 |

**为什么必须读文件而不是乘名义 bpw**：embedding 与 `output.weight` 通常保留在 Q4_K / Q6_K，imatrix 还会改变分层类型分配。8B 在 2.3125 bpw 下的纯算术下界是 2.368 GB，但真实文件通常是 2.7–3.0 GB —— 差的那部分是没被低位量化的张量。

S1 会写一个**合成 GGUF** 再读回来，逐项断言上面的字节数，这样解析器本身在动大文件之前就被验证过。


In [ ]:
# ---------------- 纯 Python GGUF 头解析（无第三方依赖） ----------------
# (name, block_size, type_size_bytes)，来自 ggml-common.h 的结构体实际大小
GGML_TYPES = {
    0:  ("F32",      1,   4),
    1:  ("F16",      1,   2),
    2:  ("Q4_0",     32,  18),
    3:  ("Q4_1",     32,  20),
    6:  ("Q5_0",     32,  22),
    7:  ("Q5_1",     32,  24),
    8:  ("Q8_0",     32,  34),
    9:  ("Q8_1",     32,  36),
    10: ("Q2_K",     256, 84),
    11: ("Q3_K",     256, 110),
    12: ("Q4_K",     256, 144),
    13: ("Q5_K",     256, 176),
    14: ("Q6_K",     256, 210),
    15: ("Q8_K",     256, 292),
    16: ("IQ2_XXS",  256, 66),
    17: ("IQ2_XS",   256, 74),
    18: ("IQ3_XXS",  256, 98),
    19: ("IQ1_S",    256, 50),
    20: ("IQ4_NL",   32,  18),
    21: ("IQ3_S",    256, 110),
    22: ("IQ2_S",    256, 82),
    23: ("IQ4_XS",   256, 136),
    24: ("I8",       1,   1),
    25: ("I16",      1,   2),
    26: ("I32",      1,   4),
    27: ("I64",      1,   8),
    28: ("F64",      1,   8),
    29: ("IQ1_M",    256, 56),
    30: ("BF16",     1,   2),
    34: ("TQ1_0",    256, 54),
    35: ("TQ2_0",    256, 66),
    36: ("MXFP4",    32,  17),
}

# 标量 value_type -> 字节数（GGUF 规范）
_GGUF_SCALAR = {0: 1, 1: 1, 2: 2, 3: 2, 4: 4, 5: 4, 6: 4, 7: 1, 10: 8, 11: 8, 12: 8}


class _Reader:
    def __init__(self, f):
        self.f = f

    def read(self, n):
        b = self.f.read(n)
        if len(b) != n:
            raise EOFError(f"unexpected EOF: want {n}, got {len(b)}")
        return b

    def u32(self):
        return struct.unpack("<I", self.read(4))[0]

    def u64(self):
        return struct.unpack("<Q", self.read(8))[0]

    def string(self):
        n = self.u64()
        return self.read(n).decode("utf-8", "replace")


def _skip_value(r, vt):
    """跳过任意 metadata value，不物化大数组（vocab 有 15 万个字符串）。"""
    if vt == 8:                                  # string
        n = r.u64()
        r.f.seek(n, 1)
    elif vt == 9:                                # array
        et = r.u32()
        cnt = r.u64()
        if et in _GGUF_SCALAR:
            r.f.seek(_GGUF_SCALAR[et] * cnt, 1)
        elif et == 8:
            for _ in range(cnt):
                n = r.u64()
                r.f.seek(n, 1)
        else:
            raise ValueError(f"不支持的数组元素类型 {et}")
    elif vt in _GGUF_SCALAR:
        r.read(_GGUF_SCALAR[vt])
    else:
        raise ValueError(f"不支持的 value type {vt}")


def gguf_scan(path, deadline=None):
    """读 GGUF 头，返回逐张量的类型/形状/真实字节。"""
    path = Path(path)
    with open(path, "rb") as f:
        r = _Reader(f)
        magic = r.read(4)
        if magic != b"GGUF":
            raise ValueError(f"不是 GGUF 文件（magic={magic!r}）: {path}")
        ver = r.u32()
        if ver not in (2, 3):
            raise ValueError(f"不支持的 GGUF 版本 {ver}")
        n_tensors = r.u64()
        n_kv = r.u64()

        WANT = {"general.architecture", "general.name", "general.file_type"}
        kv = {}
        for _ in range(n_kv):
            k = r.string()
            vt = r.u32()
            if k == "general.alignment" and vt == 4:
                kv[k] = r.u32()
            elif k in WANT and vt == 8:
                kv[k] = r.string()
            else:
                _skip_value(r, vt)

        align = int(kv.get("general.alignment", 32))
        tensors = []
        for i in range(n_tensors):
            if deadline and i % 64 == 0:
                check_deadline(deadline, "gguf_scan")
            name = r.string()
            nd = r.u32()
            dims = [r.u64() for _ in range(nd)]
            tt = r.u32()
            off = r.u64()
            tensors.append({"name": name, "dims": dims, "type_id": tt, "offset": off})
        data_start = f.tell()

    pad = (-data_start) % align
    data_start += pad

    per_type, per_class = {}, {}
    n_params = 0
    tensor_bytes = 0
    for t in tensors:
        n = 1
        for d in t["dims"]:
            n *= d
        t["n_elements"] = n
        n_params += n

        tname, blk, tsz = GGML_TYPES.get(t["type_id"], ("UNKNOWN", 1, 1))
        t["type"] = tname
        nb = ((n + blk - 1) // blk) * tsz
        t["nbytes"] = nb
        tensor_bytes += nb
        per_type[tname] = per_type.get(tname, 0) + nb

        # 粗分类，用来看哪些张量没被低位量化
        nm = t["name"].lower()
        if "token_embd" in nm or "output.weight" in nm:
            cls = "embed/head"
        elif "attn_" in nm:
            cls = "attn"
        elif "ffn_" in nm:
            cls = "ffn"
        else:
            cls = "other"
        per_class[cls] = per_class.get(cls, 0) + nb

    fsize = path.stat().st_size
    return {
        "path": str(path),
        "file_bytes": fsize,
        "file_gb": round(fsize / 2**30, 3),
        "gguf_version": ver,
        "architecture": kv.get("general.architecture"),
        "alignment": align,
        "n_tensors": len(tensors),
        "n_params": n_params,
        "tensor_bytes": tensor_bytes,
        "header_bytes": fsize - tensor_bytes,
        "bpw_effective": round(tensor_bytes * 8 / n_params, 4) if n_params else None,
        "bpw_filesize": round(fsize * 8 / n_params, 4) if n_params else None,
        "bytes_per_type": {k: v for k, v in sorted(per_type.items(), key=lambda x: -x[1])},
        "bytes_per_class": {k: v for k, v in sorted(per_class.items(), key=lambda x: -x[1])},
        "tensors": tensors,
    }


print("GGUF 解析器已加载")



In [ ]:

# ---------------- 跨阶段共享的评测工具 ----------------
# 故意放在独立的 cell（而不是某个阶段里）：这样即使你跳过中间的 cell
# 单独跑后面的阶段，也不会因为函数没定义而 NameError。

PPL_RE = re.compile(r"final estimate:\s*PPL\s*=\s*([0-9.]+)\s*\+/-\s*([0-9.]+)", re.I)


def _fit_ngl(nbytes, vram_usable_gb, margin_gb=1.0):
    """权重能不能进显存。返回 llama.cpp 的 -ngl 值。"""
    if not vram_usable_gb:
        return 0
    need = nbytes / 2**30 + margin_gb
    return 99 if need < vram_usable_gb else 0


def split_eval(path, n_groups, out_dir):
    """把评测文本切成 n_groups 份不相交的连续子文件，用于配对 bootstrap。"""
    txt = Path(path).read_text(errors="replace")
    paras = [p.strip() for p in re.split(r"\n\s*\n", txt) if p.strip()]
    if len(paras) < n_groups:
        paras = [p for p in txt.split("\n") if p.strip()]
    if not paras:
        raise ValueError(f"评测语料为空: {path}")

    # 连续等量切片：保留文档顺序，各组段落数尽量相等（字符数自然接近）
    n_groups = max(1, min(n_groups, len(paras)))
    per = math.ceil(len(paras) / n_groups)
    groups = [paras[i:i + per] for i in range(0, len(paras), per)]

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for i, g in enumerate(groups):
        fp = out_dir / f"eval_g{i:03d}.txt"
        fp.write_text("\n\n".join(g), encoding="utf-8")
        paths.append(fp)
    return paths


def run_ppl(model_path, text_path, deadline, tag):
    """跑 llama-perplexity，解析 Final estimate。返回 NLL = ln(PPL)。"""
    v = vram_gb()
    usable = (v["total"] - 1.0) if v else 0.0
    ngl = _fit_ngl(Path(model_path).stat().st_size, usable)
    cmd = [llama_bin("llama-perplexity"),
           "-m", model_path, "-f", text_path,
           "-c", CFG["NLL_CTX"],
           "--chunks", CFG["NLL_CHUNKS_PER_GROUP"],
           "-ngl", ngl, "-t", os.cpu_count() or 4,
           "-b", "512", "--seed", CFG["SEED"]]
    logf = LOGS / f"ppl_{tag}.log"
    rc, tail = run_cmd(cmd, logf, deadline, check=False)

    full = logf.read_text(errors="replace")
    m = PPL_RE.search(full)
    if not m:
        raise RuntimeError(f"{tag}: 解析不到 PPL。日志尾部:\n{tail}")
    ppl = float(m.group(1))
    return {"ppl": ppl, "ppl_err": float(m.group(2)), "nll": math.log(ppl),
            "ngl": ngl, "chunks": CFG["NLL_CHUNKS_PER_GROUP"], "ctx": CFG["NLL_CTX"]}


print("共享评测工具已加载：_fit_ngl / split_eval / run_ppl")


## S1 · 短时 smoke test

先证明解析器和预算算术是对的，再碰大文件。

In [ ]:
def gguf_write_synth(path, specs, alignment=32):
    """写一个最小但合法的 GGUF v3，用于自检解析器。specs = [(name, dims, type_id)]"""
    import io
    buf = io.BytesIO()

    def u32(x): buf.write(struct.pack("<I", x))
    def u64(x): buf.write(struct.pack("<Q", x))

    def s(x):
        b = x.encode()
        u64(len(b))
        buf.write(b)

    buf.write(b"GGUF")
    u32(3)
    u64(len(specs))
    u64(2)
    s("general.architecture"); u32(8); s("llama")
    s("general.alignment");    u32(4); u32(alignment)

    sizes = []
    cur = 0
    for name, dims, tid in specs:
        s(name)
        u32(len(dims))
        for d in dims:
            u64(d)
        u32(tid)
        u64(cur)
        n = 1
        for d in dims:
            n *= d
        _, blk, tsz = GGML_TYPES[tid]
        nb = ((n + blk - 1) // blk) * tsz
        sizes.append(nb)
        cur += ((nb + alignment - 1) // alignment) * alignment

    while len(buf.getvalue()) % alignment:
        buf.write(b"\x00")
    for nb in sizes:
        buf.write(b"\x00" * nb)

    Path(path).write_bytes(buf.getvalue())
    return Path(path)


def _torch_microbench(deadline):
    """fp16 vs bf16 矩阵乘 + 拷贝带宽。T4 上 bf16 会明显更慢，这正是要测的。"""
    out = {}
    try:
        import torch
    except Exception as e:
        return {"error": f"torch 不可用: {e}"}
    if not torch.cuda.is_available():
        return {"cuda": False, "note": "无 GPU，跳过"}

    N = 2048
    flops = 2.0 * N**3
    for dt, label in ((torch.float16, "fp16"), (torch.bfloat16, "bf16")):
        try:
            check_deadline(deadline, "S1 矩阵乘")
            a = torch.randn(N, N, device="cuda", dtype=dt)
            b = torch.randn(N, N, device="cuda", dtype=dt)
            for _ in range(2):
                c = a @ b
            torch.cuda.synchronize()
            t = time.time()
            for _ in range(5):
                c = a @ b
            torch.cuda.synchronize()
            el = (time.time() - t) / 5
            out[label] = {"ms": round(el * 1000, 3), "tflops": round(flops / el / 1e12, 2)}
            del a, b, c
            torch.cuda.empty_cache()
        except Exception as e:
            out[label] = {"error": repr(e)}

    # 拷贝带宽：读 + 写各 nbytes
    try:
        nbytes = 256 << 20
        x = torch.empty(nbytes // 4, device="cuda", dtype=torch.float32)
        y = torch.empty_like(x)
        for _ in range(2):
            y.copy_(x)
        torch.cuda.synchronize()
        t = time.time()
        for _ in range(5):
            y.copy_(x)
        torch.cuda.synchronize()
        el = (time.time() - t) / 5
        out["copy"] = {"gb_moved": round(2 * nbytes / 2**30, 3),
                       "ms": round(el * 1000, 2),
                       "gb_per_s_derived": round(2 * nbytes / el / 1e9, 1),
                       "note": "read+write 的派生值，不是 DRAM 计数器实测"}
        del x, y
        torch.cuda.empty_cache()
    except Exception as e:
        out["copy"] = {"error": repr(e)}

    free, total = torch.cuda.mem_get_info()
    out["vram_after_gb"] = {"free": round(free / 2**30, 2), "total": round(total / 2**30, 2)}
    return out


def _disk_bench(deadline, mb=256):
    p = WORK / "smoke_disk.bin"
    chunk = b"\x00" * (1 << 20)
    t = time.time()
    try:
        with open(p, "wb") as f:
            for _ in range(mb):
                f.write(chunk)
            f.flush()
            os.fsync(f.fileno())
        el = time.time() - t
        return {"mb": mb, "sec": round(el, 3), "mb_per_s": round(mb / el, 1)}
    finally:
        if p.exists():
            p.unlink()


def stage_s1_smoke(deadline):
    res = {"checks": {}, "bench": {}}

    # ---- 1. 合成 GGUF 往返 ----
    specs = [
        ("token_embd.weight",     (256, 128), 1),   # F16
        ("blk.0.ffn_gate.weight", (256, 256), 17),  # IQ2_XS
        ("blk.0.ffn_up.weight",   (256, 256), 23),  # IQ4_XS
        ("blk.0.attn_q.weight",   (256, 256), 12),  # Q4_K
        ("output.weight",         (128, 256), 0),   # F32
    ]
    sp = WORK / "smoke_synth.gguf"
    gguf_write_synth(sp, specs)
    scan = gguf_scan(sp)

    expected = {}
    for name, dims, tid in specs:
        n = 1
        for d in dims:
            n *= d
        _, blk, tsz = GGML_TYPES[tid]
        expected[name] = ((n + blk - 1) // blk) * tsz

    got = {t["name"]: t["nbytes"] for t in scan["tensors"]}
    res["checks"]["gguf_roundtrip_bytes_match"] = (got == expected)
    res["checks"]["gguf_expected"] = expected
    res["checks"]["gguf_got"] = got

    # IQ2_XS / IQ4_XS 的 bpw 必须精确等于名义值
    def bpw_of(name):
        t = [x for x in scan["tensors"] if x["name"] == name][0]
        return t["nbytes"] * 8 / t["n_elements"]

    res["checks"]["bpw_iq2_xs"] = round(bpw_of("blk.0.ffn_gate.weight"), 6)
    res["checks"]["bpw_iq4_xs"] = round(bpw_of("blk.0.ffn_up.weight"), 6)
    res["checks"]["bpw_iq2_xs_ok"] = abs(res["checks"]["bpw_iq2_xs"] - 2.3125) < 1e-6
    res["checks"]["bpw_iq4_xs_ok"] = abs(res["checks"]["bpw_iq4_xs"] - 4.25) < 1e-6
    res["checks"]["synth_file_mb"] = round(sp.stat().st_size / 2**20, 4)

    # ---- 2. GPU 微基准 ----
    res["bench"]["torch"] = _torch_microbench(deadline)

    # ---- 3. 磁盘 ----
    res["bench"]["disk"] = _disk_bench(deadline)

    # ---- 4. 预算换算 ----
    w_q2_gb = CFG["PARAM_COUNT"] * 2.3125 / 8 / 2**30
    w_q4_gb = CFG["PARAM_COUNT"] * 4.25 / 8 / 2**30
    w_f16_gb = CFG["PARAM_COUNT"] * 2 / 2**30
    res["budget_math"] = {
        "weights_iq2_xs_gib_lower_bound": round(w_q2_gb, 3),
        "weights_iq4_xs_gib_lower_bound": round(w_q4_gb, 3),
        "weights_f16_gib": round(w_f16_gb, 3),
        "note": "GiB 下界，不含 embed/head 的高精度保留；真实字节由 S5 从文件读",
        "nll_tokens_planned": CFG["NLL_GROUPS"] * CFG["NLL_CHUNKS_PER_GROUP"] * CFG["NLL_CTX"],
    }

    ok = all(v for k, v in res["checks"].items() if isinstance(v, bool))
    res["smoke_ok"] = ok

    print("=" * 74)
    print("smoke checks:")
    for k, v in res["checks"].items():
        print(f"  {k:34s} {v}")
    print("bench:", json.dumps(res["bench"], indent=1, ensure_ascii=False))
    print("budget:", json.dumps(res["budget_math"], indent=1, ensure_ascii=False))
    print(f"SMOKE_OK = {ok}")
    print("=" * 74)

    if not ok:
        raise RuntimeError("smoke 自检失败，先修解析器再用大文件")
    return res


s1 = run_stage("S1_SMOKE", stage_s1_smoke)


---

## 从 S2 开始：需要你手动挂载文件

S2 之后每个阶段都会先做 `need(...)` 检查，缺路径直接 `GuardError`，**不会**去下载。

### S2_BUILD

从源码构建。Kaggle 镜像自带 `cmake` / `g++` / `nvcc`，不需要 `pip install` 任何东西。

```
cmake -B build -S <src> -DGGML_NATIVE=ON -DGGML_CUDA={ON|OFF} -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release
cmake --build build --config Release -j 4 --target llama-imatrix llama-quantize llama-perplexity llama-cli
```

`LLAMA_CURL=OFF` 是必要的：否则它会尝试链接 libcurl 并可能去拉网络资源。

`CFG["BUILD_CUDA"]=False` 时只构建 CPU 版 —— **imatrix 和 quantize 都是 CPU 路径，不需要 GPU**。只有 S6/S7 需要 CUDA 才有速度。

### S3_IMATRIX

```
llama-imatrix -m <base.gguf> -f <calib.txt> -o imatrix.gguf -c 512 --chunks 32 -ngl <n>
```

`-ngl` 由**显存拟合判定**自动决定：`base 字节 + 1.0 GB < 可用显存` 就 `-ngl 99`，否则 `-ngl 0`（纯 CPU）。

这就是为什么 8B 的 f16 base 在 T4 上会退到 CPU：15.25 GiB 权重放不进约 15.0 GiB 的可用显存（还没算 KV 与 CUDA context）。**Q8_0 base（约 8.1 GiB）能进显存，是本机推荐路径。**

### S4_QUANT

```
llama-quantize --imatrix imatrix.gguf <base.gguf> <out_iq2xs.gguf> IQ2_XS <threads>
llama-quantize --imatrix imatrix.gguf <base.gguf> <out_iq4xs.gguf> IQ4_XS <threads>
```

两个都用 `--imatrix`。`IQ4_XS` 不要求它，但**必须给**，否则比较不公平。IQ2_XS 若失败，报错信息里会明确指向 imatrix 缺失。


In [ ]:
def _find_llama_src():
    if CFG["LLAMA_SRC"]:
        p = Path(CFG["LLAMA_SRC"])
        if (p / "CMakeLists.txt").exists():
            return p
        raise GuardError(f"CFG['LLAMA_SRC'] 下没有 CMakeLists.txt: {p}")
    for cand in [Path("/kaggle/input/llama.cpp"),
                 Path("/kaggle/input/llama-cpp/llama.cpp"),
                 Path("/kaggle/input/llamacpp/llama.cpp")]:
        if (cand / "CMakeLists.txt").exists():
            return cand
    if CFG["ALLOW_NETWORK"]:
        dst = WORK / "llama.cpp"
        if not (dst / "CMakeLists.txt").exists():
            run_cmd(["git", "clone", "--depth", "1",
                     "https://github.com/ggml-org/llama.cpp", str(dst)],
                    LOGS / "s2_clone.log", time.time() + 900)
        return dst
    raise GuardError(
        "找不到 llama.cpp 源码。请把源码挂到 /kaggle/input/llama.cpp，"
        "或设置 CFG['LLAMA_SRC']，或显式打开 CFG['ALLOW_NETWORK']=True 允许 clone。")


def stage_s2_build(deadline):
    src = _find_llama_src()
    build = src / "build"
    log(f"源码: {src}")

    # 已有构建就直接复用
    have = {n: (build / "bin" / n) for n in
            ["llama-imatrix", "llama-quantize", "llama-perplexity", "llama-cli"]}
    if all(p.exists() for p in have.values()):
        log("已有完整构建，跳过编译")
        return {"src": str(src), "build": str(build), "reused": True,
                "bins": {k: str(v) for k, v in have.items()}}

    cmake_args = [
        "cmake", "-B", str(build), "-S", str(src),
        "-DCMAKE_BUILD_TYPE=Release",
        "-DGGML_NATIVE=ON",
        f"-DGGML_CUDA={'ON' if CFG['BUILD_CUDA'] else 'OFF'}",
        "-DLLAMA_CURL=OFF",
        "-DLLAMA_BUILD_TESTS=OFF",
        "-DLLAMA_BUILD_EXAMPLES=OFF",
    ]
    run_cmd(cmake_args, LOGS / "s2_cmake.log", deadline)

    run_cmd(["cmake", "--build", str(build), "--config", "Release",
             "-j", str(CFG["BUILD_JOBS"]), "--target",
             "llama-imatrix", "llama-quantize", "llama-perplexity", "llama-cli"],
            LOGS / "s2_build.log", deadline)

    bins = {}
    for n in have:
        p = build / "bin" / n
        if not p.exists():
            p2 = build / n
            p = p2 if p2.exists() else p
        bins[n] = str(p) if p.exists() else None

    missing = [k for k, v in bins.items() if v is None]
    if missing:
        raise RuntimeError(f"编译结束但缺少可执行文件: {missing}（看 {LOGS}/s2_build.log）")

    ver = subprocess.run([bins["llama-cli"], "--version"],
                         capture_output=True, text=True).stdout.strip()[:300]
    return {"src": str(src), "build": str(build), "reused": False,
            "bins": bins, "version": ver, "cuda": CFG["BUILD_CUDA"]}


s2 = run_stage("S2_BUILD", stage_s2_build)


In [ ]:
def stage_s3_imatrix(deadline):
    base = need("BASE_GGUF")
    calib = need("CALIB_TXT")
    if not CFG["BASE_KIND"]:
        raise GuardError("CFG['BASE_KIND'] 未设置（必须写 f16 / bf16 / q8_0 ...）")

    # 校准语料必须足够长，否则 imatrix 没有意义
    calib_chars = calib.stat().st_size
    if calib_chars < 200_000:
        log(f"!! 校准语料只有 {calib_chars/1024:.0f} KB，imatrix 可能不稳")

    out_path = Path(CFG["IMATRIX_GGUF"]) if CFG["IMATRIX_GGUF"] else (MDLS / "imatrix.gguf")

    v = vram_gb()
    usable = (v["total"] - 1.0) if v else 0.0
    ngl = _fit_ngl(base.stat().st_size, usable)
    log(f"base={base.name} {base.stat().st_size/2**30:.2f} GB, 可用显存 {usable:.1f} GB → -ngl {ngl}")

    cmd = [llama_bin("llama-imatrix"),
           "-m", base, "-f", calib, "-o", out_path,
           "-c", CFG["IMATRIX_CTX"], "--chunks", CFG["IMATRIX_CHUNKS"],
           "-ngl", ngl, "-t", os.cpu_count() or 4]
    rc, tail = run_cmd(cmd, LOGS / "s3_imatrix.log", deadline, check=False)

    if not out_path.exists() or out_path.stat().st_size == 0:
        raise RuntimeError(f"imatrix 未生成（rc={rc}）。日志尾部:\n{tail}")

    rec = {
        "imatrix": str(out_path),
        "bytes": out_path.stat().st_size,
        "sha256": sha256_of(out_path, deadline) if CFG["HASH_GGUF"] else None,
        "base_gguf": str(base),
        "base_kind": CFG["BASE_KIND"],
        "base_bytes": base.stat().st_size,
        "calib_txt": str(calib),
        "calib_bytes": calib_chars,
        "calib_source": CFG["CALIB_SOURCE"],
        "ctx": CFG["IMATRIX_CTX"],
        "chunks": CFG["IMATRIX_CHUNKS"],
        "ngl": ngl,
        "on_gpu": ngl > 0,
        "rc": rc,
    }
    print(json.dumps(rec, indent=1, ensure_ascii=False))
    return rec


s3 = run_stage("S3_IMATRIX", stage_s3_imatrix)


In [ ]:
FORMATS = ["IQ2_XS", "IQ4_XS"]


def stage_s4_quant(deadline):
    base = need("BASE_GGUF")
    imat = Path(CFG["IMATRIX_GGUF"]) if CFG["IMATRIX_GGUF"] else (MDLS / "imatrix.gguf")
    if not imat.exists():
        raise GuardError(f"找不到 imatrix: {imat}。IQ2_XS 没有 imatrix 会直接 bail out，不做降级。")

    qbin = llama_bin("llama-quantize")
    threads = os.cpu_count() or 4
    out = {}

    for fmt in FORMATS:
        check_deadline(deadline, f"S4_QUANT {fmt}")
        dst = MDLS / f"qwen3-8b-{fmt}.gguf"
        if dst.exists() and dst.stat().st_size > 0 and not CFG["FORCE_RERUN"]:
            log(f"{fmt} 已存在，跳过（{dst.stat().st_size/2**30:.2f} GB）")
        else:
            cmd = [qbin, "--imatrix", imat, base, dst, fmt, threads]
            rc, tail = run_cmd(cmd, LOGS / f"s4_quant_{fmt}.log", deadline, check=False)
            if not dst.exists() or dst.stat().st_size == 0:
                hint = ""
                if "imatrix" in tail.lower() or "garbage" in tail.lower():
                    hint = "  ← 这是 imatrix 缺失/无效触发的 bail out"
                raise RuntimeError(f"{fmt} 量化失败（rc={rc}）。{hint}\n--- tail ---\n{tail}")

        # 立即验一遍文件头，避免把坏文件带进后续阶段
        sc = gguf_scan(dst, deadline)
        dom = max(sc["bytes_per_type"].items(), key=lambda x: x[1])[0]
        out[fmt] = {
            "path": str(dst),
            "file_bytes": sc["file_bytes"],
            "file_gb": sc["file_gb"],
            "n_params": sc["n_params"],
            "bpw_effective": sc["bpw_effective"],
            "dominant_type": dom,
            "sha256": sha256_of(dst, deadline) if CFG["HASH_GGUF"] else None,
        }
        log(f"{fmt}: {sc['file_gb']:.2f} GB, {sc['bpw_effective']:.4f} bpw, 主导类型 {dom}")

    # 公平性自检：两个文件必须来自同一 base、同一 imatrix
    out["_fairness"] = {
        "same_base": True,
        "base_gguf": str(base),
        "base_kind": CFG["BASE_KIND"],
        "same_imatrix": str(imat),
        "imatrix_sha256": s3.get("sha256") if isinstance(s3, dict) else None,
        "note": "两个格式共用同一 base 与同一 imatrix；若这一条不成立，S6 的配对比较无效",
    }
    print(json.dumps(out, indent=1, ensure_ascii=False))
    return out


s4 = run_stage("S4_QUANT", stage_s4_quant)


In [ ]:
def _fmt_bytes(n):
    return f"{n/2**30:.3f} GB"


def stage_s5_bytes(deadline):
    """对 S4 的产物 + base 做逐张量类型的字节分解。"""
    targets = {}
    if CFG["BASE_GGUF"] and Path(CFG["BASE_GGUF"]).exists():
        targets[f"base_{CFG['BASE_KIND'] or 'unknown'}"] = Path(CFG["BASE_GGUF"])
    for fmt in FORMATS:
        p = MDLS / f"qwen3-8b-{fmt}.gguf"
        if p.exists():
            targets[fmt] = p
    if not targets:
        raise GuardError("没有可分析的 GGUF。先跑 S4，或设置 CFG['BASE_GGUF']。")

    rows, detail = [], {}
    for name, p in targets.items():
        check_deadline(deadline, f"S5 {name}")
        sc = gguf_scan(p, deadline)
        detail[name] = {
            "file_bytes": sc["file_bytes"],
            "file_gb": sc["file_gb"],
            "n_params": sc["n_params"],
            "bpw_effective": sc["bpw_effective"],
            "bpw_filesize": sc["bpw_filesize"],
            "bytes_per_type": sc["bytes_per_type"],
            "bytes_per_class": sc["bytes_per_class"],
            "header_bytes": sc["header_bytes"],
            "n_tensors": sc["n_tensors"],
        }
        rows.append((name, sc["file_gb"], sc["bpw_effective"],
                     sc["bytes_per_class"].get("embed/head", 0)))

    print("=" * 78)
    print(f"{'name':22s} {'file GB':>9s} {'bpw':>8s} {'embed+head GB':>14s}")
    print("-" * 78)
    for name, gb, bpw, eh in rows:
        bpw_s = f"{bpw:8.4f}" if bpw is not None else f"{'n/a':>8s}"
        print(f"{name:22s} {gb:9.3f} {bpw_s} {eh/2**30:14.3f}")
    print("-" * 78)
    for name in detail:
        d = detail[name]
        print(f"{name} 按类型:", {k: round(v / 2**30, 3) for k, v in d["bytes_per_type"].items()})
        if d["n_params"] == 0:
            print(f"  !! {name} 没有张量（只有词表/元数据），bpw 无法计算")
    print("=" * 78)

    # 相对压缩比（只对同时存在的两个格式算）
    ratios = {}
    if "IQ2_XS" in detail and "IQ4_XS" in detail:
        b2 = detail["IQ2_XS"]["file_bytes"]
        b4 = detail["IQ4_XS"]["file_bytes"]
        ratios["iq4_over_iq2"] = round(b4 / b2, 4)
        ratios["bytes_saved_pct"] = round((1 - b2 / b4) * 100, 2)
        # 只算权重张量（去掉 header）更干净
        t2 = detail["IQ2_XS"]["file_bytes"] - detail["IQ2_XS"]["header_bytes"]
        t4 = detail["IQ4_XS"]["file_bytes"] - detail["IQ4_XS"]["header_bytes"]
        ratios["tensor_only_iq4_over_iq2"] = round(t4 / t2, 4)
        ratios["note"] = "这些是文件字节比，不是端到端加速比"
        print("IQ4_XS / IQ2_XS 字节比:", ratios)

    return {"per_model": detail, "ratios": ratios}


s5 = run_stage("S5_BYTES", stage_s5_bytes)


## S6 · 配对 NLL

同一个 base、同一份 imatrix、同一批 token 窗口，逐组配对。

In [ ]:
def stage_s6_nll(deadline):
    eval_txt = need("EVAL_TXT")
    if not CFG["EVAL_SOURCE"]:
        log("!! CFG['EVAL_SOURCE'] 未设置，provenance 里会缺这一项")

    # ---- 校准/评测隔离断言（内容级）----
    isolation = {"calib": None, "eval": None, "overlap_lines": None, "ok": None}
    if CFG["CALIB_TXT"] and Path(CFG["CALIB_TXT"]).exists():
        c = Path(CFG["CALIB_TXT"]).read_text(errors="replace")
        e = eval_txt.read_text(errors="replace")
        cset = {ln.strip()[:200] for ln in c.split("\n") if len(ln.strip()) > 40}
        eset = {ln.strip()[:200] for ln in e.split("\n") if len(ln.strip()) > 40}
        ov = cset & eset
        isolation.update(
            calib=str(CFG["CALIB_TXT"]), calib_source=CFG["CALIB_SOURCE"],
            eval=str(eval_txt), eval_source=CFG["EVAL_SOURCE"],
            calib_lines=len(cset), eval_lines=len(eset),
            overlap_lines=len(ov), ok=(len(ov) == 0))
        if ov:
            raise RuntimeError(
                f"校准集与评测集有 {len(ov)} 行重叠（前 200 字符比对）。"
                "必须换 split 后再跑，否则 NLL 会被 imatrix 污染。")
        log(f"隔离检查通过：calib {len(cset)} 行 / eval {len(eset)} 行 / 重叠 0")
    else:
        isolation["ok"] = "unknown"
        log("!! 没有校准语料路径，无法做隔离断言 —— 结论里必须标注这一点")

    models = {}
    for fmt in FORMATS:
        p = MDLS / f"qwen3-8b-{fmt}.gguf"
        if p.exists():
            models[fmt] = p
    if len(models) < 2:
        raise GuardError(f"需要两个格式的文件，现有 {list(models)}。先跑 S4。")

    groups = split_eval(eval_txt, CFG["NLL_GROUPS"], WORK / "eval_groups")
    log(f"评测集切成 {len(groups)} 组，每组约 {groups[0].stat().st_size/1024:.0f} KB")

    per_model = {}
    for fmt, mp in models.items():
        per_model[fmt] = []
        for i, g in enumerate(groups):
            check_deadline(deadline, f"S6 {fmt} g{i}")
            r = run_ppl(mp, g, deadline, f"{fmt}_g{i}")
            r["group"] = i
            per_model[fmt].append(r)
            log(f"  {fmt} g{i}: PPL={r['ppl']:.4f} NLL={r['nll']:.4f}")

    # 配对差值：IQ2_XS - IQ4_XS（同一组、同一 token 窗口）
    d = [per_model["IQ2_XS"][i]["nll"] - per_model["IQ4_XS"][i]["nll"]
         for i in range(len(groups))]
    stats = paired_bootstrap(d)

    agg = {fmt: {"mean_nll": sum(x["nll"] for x in v) / len(v),
                 "mean_ppl": sum(x["ppl"] for x in v) / len(v),
                 "nll_sd": (sum((x["nll"] - sum(y["nll"] for y in v) / len(v))**2
                                for x in v) / max(1, len(v) - 1)) ** 0.5}
           for fmt, v in per_model.items()}

    out = {
        "isolation": isolation,
        "eval_source": CFG["EVAL_SOURCE"],
        "groups": len(groups),
        "per_group": per_model,
        "aggregate": agg,
        "paired_delta_nll_iq2_minus_iq4": stats,
        "deltas": [round(x, 6) for x in d],
        "provenance": {
            "base_gguf": CFG["BASE_GGUF"], "base_kind": CFG["BASE_KIND"],
            "imatrix": str(MDLS / "imatrix.gguf"),
            "self_quantized": True, "fair_comparison": True,
            "note": "两个文件同 base、同 imatrix、同 token 窗口",
        },
    }

    print("=" * 78)
    for fmt, a in agg.items():
        print(f"{fmt:10s} mean NLL={a['mean_nll']:.4f}  mean PPL={a['mean_ppl']:.4f}  sd={a['nll_sd']:.4f}")
    print(f"配对差 (IQ2 - IQ4): mean={stats['mean']:+.4f}  "
          f"CI95={stats['ci95']}  n={stats['n']}")
    print("=" * 78)

    # 方向性判读，但不给绝对阈值
    ci = stats.get("ci95")
    if ci and ci[0] > 0:
        verdict = "IQ2_XS 显著差于 IQ4_XS（CI 全在 0 以上）"
    elif ci and ci[1] < 0:
        verdict = "IQ2_XS 显著好于 IQ4_XS（异常，先查口径）"
    else:
        verdict = "CI 跨 0 —— 未测出差异，不等于等价"
    out["verdict"] = verdict
    print("判读:", verdict)
    return out


s6 = run_stage("S6_NLL", stage_s6_nll)


In [ ]:
QUALITY_PROMPTS_DEFAULT = [
    "The capital of France is",
    "def fibonacci(n):",
    "In 1969, the first humans landed on",
    "The chemical formula for table salt is",
    "Q: What is 17 * 23?\nA:",
    "Once upon a time, in a small village by the sea,",
    "The three primary colors of light are",
    "Translate 'good morning' into Chinese:",
]


def probe_help(binary):
    try:
        p = subprocess.run([str(binary), "--help"], capture_output=True, text=True, timeout=90)
        return (p.stdout or "") + (p.stderr or "")
    except Exception:
        return ""


def _cli_flags():
    """llama-cli 的 flag 名跨版本变过，这里探测而不是硬编码。"""
    h = probe_help(llama_bin("llama-cli"))
    flags = {}
    flags["no_conv"] = ("--no-conversation" if "--no-conversation" in h
                        else ("-no-cnv" if "-no-cnv" in h else None))
    flags["no_display_prompt"] = "--no-display-prompt" if "--no-display-prompt" in h else None
    flags["temp"] = "--temp" if "--temp" in h else "-temp"
    flags["seed"] = "--seed" if "--seed" in h else "-s"
    flags["n_predict"] = "-n" if "-n," in h or "-n " in h else "--n-predict"
    flags["simple_io"] = "--simple-io" if "--simple-io" in h else None
    return flags


def repeat_ratio(text, n=4):
    """4-gram 重复占比，用来发现退化。"""
    toks = text.split()
    if len(toks) < n + 1:
        return 0.0
    grams = [tuple(toks[i:i + n]) for i in range(len(toks) - n + 1)]
    return 1.0 - len(set(grams)) / len(grams)


def stage_s7_quality(deadline):
    flags = _cli_flags()
    log(f"llama-cli flags: {flags}")
    cli = llama_bin("llama-cli")

    models = {f: MDLS / f"qwen3-8b-{f}.gguf" for f in FORMATS
              if (MDLS / f"qwen3-8b-{f}.gguf").exists()}
    if len(models) < 2:
        raise GuardError("需要两个格式的文件，先跑 S4")

    prompts = QUALITY_PROMPTS_DEFAULT[:CFG["QUALITY_PROMPTS"]]
    v = vram_gb()
    usable = (v["total"] - 1.0) if v else 0.0

    per_model = {}
    for fmt, mp in models.items():
        ngl = _fit_ngl(mp.stat().st_size, usable)
        per_model[fmt] = []
        for i, pr in enumerate(prompts):
            check_deadline(deadline, f"S7 {fmt} p{i}")
            cmd = [cli, "-m", mp, "-p", pr, "-n", CFG["QUALITY_N_PREDICT"],
                   flags["temp"], "0", flags["seed"], CFG["SEED"],
                   "-ngl", ngl, "-t", os.cpu_count() or 4]
            if flags["no_conv"]:
                cmd.append(flags["no_conv"])
            if flags["no_display_prompt"]:
                cmd.append(flags["no_display_prompt"])
            if flags["simple_io"]:
                cmd.append(flags["simple_io"])
            logf = LOGS / f"s7_{fmt}_p{i}.log"
            rc, tail = run_cmd(cmd, logf, deadline, check=False)
            txt = logf.read_text(errors="replace")
            # 去掉可能的 prompt 回显
            body = txt.split(pr, 1)[1] if pr in txt else txt
            body = body.replace("llama_perf", "\nllama_perf").split("\nllama_perf")[0].strip()
            per_model[fmt].append({"prompt": pr, "rc": rc,
                                   "text": body[:400],
                                   "repeat4": round(repeat_ratio(body), 4),
                                   "chars": len(body)})

    # 配对：同 prompt 上两格式的输出是否一致 / 首个分歧位置
    pairs = []
    for i, pr in enumerate(prompts):
        a = per_model["IQ2_XS"][i]["text"]
        b = per_model["IQ4_XS"][i]["text"]
        same = a == b
        k = 0
        while k < min(len(a), len(b)) and a[k] == b[k]:
            k += 1
        pairs.append({"prompt": pr, "identical": same, "first_divergence_char": k,
                      "len_iq2": len(a), "len_iq4": len(b)})

    n_id = sum(1 for p in pairs if p["identical"])
    out = {
        "n_prompts": len(prompts),
        "identical_outputs": n_id,
        "identical_rate": round(n_id / len(prompts), 4) if prompts else None,
        "pairs": pairs,
        "per_model": per_model,
        "degenerate": {fmt: round(sum(x["repeat4"] for x in v2) / len(v2), 4)
                       for fmt, v2 in per_model.items()},
        "limits": [
            "贪心解码 + 32 token，样本量很小，只能发现整体崩塌，不能排序细微质量",
            "不是标准任务评测（GSM8K / HumanEval 之类），绝对分数不构成能力证据",
            "repeat4 只是退化信号，不是质量分数",
        ],
    }
    print("=" * 78)
    print(f"完全一致的输出: {n_id}/{len(prompts)}  ({out['identical_rate']})")
    print(f"平均 repeat4: {out['degenerate']}")
    for p in pairs:
        print(f"  div@{p['first_divergence_char']:4d}  {p['prompt'][:44]}")
    print("=" * 78)
    return out


s7 = run_stage("S7_QUALITY", stage_s7_quality)


## S8 · 高精度锚点（预留接口）

**单张 T4 16 GB 装不下完整 8B 高精度权重。** 这个阶段默认返回一条 `status=skipped` 记录，写明所需与可用的具体 GB 数。

In [ ]:
def stage_s8_bf16_anchor(deadline):
    """高精度锚点。这是**预留接口**，默认在 T4 上会被显存判定拒绝。"""
    anchor = CFG["BASE_GGUF"]
    if not anchor or not Path(anchor).exists():
        raise GuardError("CFG['BASE_GGUF'] 未设置 —— 锚点就是 base GGUF 本身")

    ap = Path(anchor)
    need_bytes = ap.stat().st_size + 1 << 30   # 权重 + 运行时
    v = vram_gb()
    total = v["total"] if v else 0.0
    usable = max(0.0, total - 1.0)

    decision = {
        "anchor_path": str(ap),
        "anchor_kind": CFG["BASE_KIND"],
        "anchor_gb": round(ap.stat().st_size / 2**30, 3),
        "vram_total_gb": round(total, 2),
        "vram_usable_gb": round(usable, 2),
        "required_gb": round(need_bytes / 2**30, 3),
        "fits_on_gpu": need_bytes / 2**30 < usable,
    }

    # T4 没有原生 bf16，锚点必须用 f16；把偏差写进记录
    if CFG["BASE_KIND"] == "bf16":
        decision["dtype_deviation"] = ("base 是 bf16，但本机 GPU 无原生 bf16 支持，"
                                       "llama.cpp 会走转换路径。建议改用 f16 锚点并在此记录偏差。")
    else:
        decision["dtype_deviation"] = (f"锚点 dtype = {CFG['BASE_KIND']}，"
                                       "与论文常用的 BF16 锚点不完全一致，跨平台比较时必须声明。")

    if not decision["fits_on_gpu"] and not CFG["ALLOW_CPU_ANCHOR"]:
        decision.update(
            status="skipped",
            reason=(f"需要 {decision['required_gb']} GB 显存，可用 {decision['vram_usable_gb']} GB。"
                    "单张 T4 16GB 装不下完整 8B 高精度锚点，按设计主动跳过。"
                    "要跑请换 A100 40/80，或显式打开 CFG['ALLOW_CPU_ANCHOR']=True（会非常慢）。"),
            how_to_enable=["换 40 GB 以上的卡", "或 CFG['ALLOW_CPU_ANCHOR']=True 用 CPU 跑"])
        print(json.dumps(decision, indent=1, ensure_ascii=False))
        return decision

    # 真正执行锚点
    eval_txt = need("EVAL_TXT")
    groups = split_eval(eval_txt, CFG["NLL_GROUPS"], WORK / "eval_groups")
    vals = []
    for i, g in enumerate(groups):
        check_deadline(deadline, f"S8 g{i}")
        r = run_ppl(ap, g, deadline, f"anchor_g{i}")
        r["group"] = i
        vals.append(r)
        log(f"  anchor g{i}: PPL={r['ppl']:.4f}")

    decision.update(
        status="ok",
        per_group=vals,
        mean_nll=sum(x["nll"] for x in vals) / len(vals),
        mean_ppl=sum(x["ppl"] for x in vals) / len(vals),
        note="锚点 NLL 用于计算 rho = dNLL(Q2,Q4) / dNLL(Q4,anchor)",
    )
    print(json.dumps({k: v2 for k, v2 in decision.items() if k != "per_group"},
                     indent=1, ensure_ascii=False))
    return decision


s8 = run_stage("S8_BF16_ANCHOR", stage_s8_bf16_anchor)


In [ ]:
def stage_sx_explore(deadline):
    """第三方现成量化：只看字节和 NLL，**永远不进入公平比较表**。"""
    p = need("EXPLORE_GGUF")
    sc = gguf_scan(p, deadline)
    rec = {
        "path": str(p),
        "file_gb": sc["file_gb"],
        "bpw_effective": sc["bpw_effective"],
        "bytes_per_type": sc["bytes_per_type"],
        "provenance": {
            "self_quantized": False,
            "fair_comparison": False,
            "why_not": ("第三方 GGUF 的校准集、imatrix、量化脚本、llama.cpp 版本全部未知。"
                        "它的 NLL 只能说明这个文件本身如何，不能与自建 IQ2_XS/IQ4_XS 做公平比较。"),
            "allowed_use": "探索性测试：确认格式可加载、量级是否合理、有没有明显崩塌",
        },
    }

    if CFG["EVAL_TXT"] and Path(CFG["EVAL_TXT"]).exists():
        eval_txt = Path(CFG["EVAL_TXT"])
        groups = split_eval(eval_txt, CFG["NLL_GROUPS"], WORK / "eval_groups")
        vals = []
        for i, g in enumerate(groups):
            check_deadline(deadline, f"SX g{i}")
            r = run_ppl(p, g, deadline, f"explore_g{i}")
            r["group"] = i
            vals.append(r)
        rec["mean_nll"] = sum(x["nll"] for x in vals) / len(vals)
        rec["mean_ppl"] = sum(x["ppl"] for x in vals) / len(vals)
        rec["per_group"] = vals
    else:
        rec["mean_nll"] = None
        rec["note"] = "没有评测语料，只做了字节分析"

    print(json.dumps({k: v2 for k, v2 in rec.items() if k != "per_group"},
                     indent=1, ensure_ascii=False))
    return rec


sx = run_stage("SX_EXPLORE", stage_sx_explore)


## S9 · 汇总报告

In [ ]:
def _get(stage):
    r = STATE.get(stage) or {}
    return r.get("result") if r.get("status") == "ok" else None


def stage_s9_report(deadline):
    assert THIRD_PARTY_IN_COMPARISON is False, "第三方量化禁止进入公平比较表"

    env = _get("S0_ENV")
    smoke = _get("S1_SMOKE")
    s3r = _get("S3_IMATRIX")
    s4r = _get("S4_QUANT")
    s5r = _get("S5_BYTES")
    s6r = _get("S6_NLL")
    s7r = _get("S7_QUALITY")
    s8r = _get("S8_BF16_ANCHOR")
    sxr = _get("SX_EXPLORE")

    L = []
    L.append("# transformer-residual-lab · Kaggle 运行报告\n")
    L.append(f"生成时间（UTC）：{datetime.now(timezone.utc).isoformat(timespec='seconds')}")
    L.append(f"模型：`{CFG['MODEL_ID']}`（{CFG['PARAM_COUNT']/1e9:.2f} B 参数）\n")

    # ---- 环境 ----
    L.append("## 1. 环境\n")
    if env:
        L.append(f"- 位置：{env['where']}，Python {env['python']}")
        L.append(f"- CPU {env['cpu_count']} 核，RAM {env['ram']}")
        for g in env.get("gpus", []):
            L.append(f"- GPU{g['idx']}：{g['name']}，CC{g['cc']}，{g['total_gb']} GB，"
                     f"native_bf16={g['native_bf16']}")
        L.append(f"- 可用显存估计：{env['vram_usable_gb_est']} GB")
        for k, v2 in env.get("disks_gb", {}).items():
            L.append(f"- 磁盘 `{k}`：free {v2.get('free')} GB")
        for w in env.get("warnings", []):
            L.append(f"- **注意**：{w}")
    else:
        L.append("_S0_ENV 未运行_")
    L.append("")

    # ---- smoke ----
    L.append("## 2. Smoke test\n")
    if smoke:
        L.append(f"- SMOKE_OK = **{smoke['smoke_ok']}**")
        L.append(f"- IQ2_XS bpw 自检：{smoke['checks'].get('bpw_iq2_xs')}（应为 2.3125）")
        L.append(f"- IQ4_XS bpw 自检：{smoke['checks'].get('bpw_iq4_xs')}（应为 4.25）")
        tb = smoke.get("bench", {}).get("torch", {})
        if "fp16" in tb and "bf16" in tb:
            r = tb["bf16"]["tflops"] / tb["fp16"]["tflops"] if tb["fp16"]["tflops"] else None
            L.append(f"- fp16 {tb['fp16']['tflops']} TFLOP/s vs bf16 {tb['bf16']['tflops']} TFLOP/s"
                     + (f"（比值 {r:.2f}）" if r else ""))
        if "copy" in tb:
            L.append(f"- 拷贝带宽（派生值）：{tb['copy']['gb_per_s_derived']} GB/s")
        d = smoke.get("bench", {}).get("disk", {})
        if d:
            L.append(f"- 磁盘写：{d.get('mb_per_s')} MB/s")
    else:
        L.append("_S1_SMOKE 未运行_")
    L.append("")

    # ---- 字节 ----
    L.append("## 3. 实际字节\n")
    if s5r:
        L.append("| 文件 | 文件 GB | 有效 bpw | embed+head GB |")
        L.append("|---|---|---|---|")
        for name, d in s5r["per_model"].items():
            eh = d["bytes_per_class"].get("embed/head", 0) / 2**30
            L.append(f"| {name} | {d['file_gb']:.3f} | {d['bpw_effective']:.4f} | {eh:.3f} |")
        if s5r.get("ratios"):
            L.append("")
            L.append(f"字节比（IQ4/IQ2）：{s5r['ratios'].get('iq4_over_iq2')}，"
                     f"节省 {s5r['ratios'].get('bytes_saved_pct')}%")
    else:
        L.append("_S5_BYTES 未运行_")
    L.append("")

    # ---- NLL ----
    L.append("## 4. 配对 NLL\n")
    if s6r:
        L.append(f"- 评测语料来源：{s6r.get('eval_source') or '未声明'}")
        iso = s6r.get("isolation", {})
        L.append(f"- 校准/评测隔离：{iso.get('ok')}（重叠 {iso.get('overlap_lines')} 行）")
        L.append(f"- 分组数：{s6r['groups']}，每组 {CFG['NLL_CHUNKS_PER_GROUP']} chunk × "
                 f"{CFG['NLL_CTX']} ctx")
        L.append("")
        L.append("| 格式 | 平均 NLL | 平均 PPL | NLL 标准差 |")
        L.append("|---|---|---|---|")
        for fmt, a in s6r["aggregate"].items():
            L.append(f"| {fmt} | {a['mean_nll']:.4f} | {a['mean_ppl']:.4f} | {a['nll_sd']:.4f} |")
        st = s6r["paired_delta_nll_iq2_minus_iq4"]
        L.append("")
        L.append(f"配对差（IQ2 − IQ4）：均值 **{st['mean']:+.4f} nats**，"
                 f"95% CI {st['ci95']}，n={st['n']}")
        L.append(f"\n判读：{s6r.get('verdict')}")
    else:
        L.append("_S6_NLL 未运行_")
    L.append("")

    # ---- 质量 ----
    L.append("## 5. 有限配对质量检查\n")
    if s7r:
        L.append(f"- 贪心续写完全一致：{s7r['identical_outputs']}/{s7r['n_prompts']}"
                 f"（{s7r['identical_rate']}）")
        L.append(f"- 平均 repeat4（越低越好）：{s7r['degenerate']}")
        for lim in s7r["limits"]:
            L.append(f"- 限制：{lim}")
    else:
        L.append("_S7_QUALITY 未运行_")
    L.append("")

    # ---- 锚点 ----
    L.append("## 6. 高精度锚点\n")
    if s8r:
        L.append(f"- 状态：**{s8r.get('status')}**")
        if s8r.get("status") == "skipped":
            L.append(f"- 原因：{s8r.get('reason')}")
        else:
            L.append(f"- 锚点 NLL：{s8r.get('mean_nll'):.4f}")
            if s6r:
                d4 = s8r["mean_nll"] - s6r["aggregate"]["IQ4_XS"]["mean_nll"]
                d2 = s8r["mean_nll"] - s6r["aggregate"]["IQ2_XS"]["mean_nll"]
                if abs(d4) > 1e-9:
                    L.append(f"- rho = dNLL(Q2,Q4) / dNLL(Q4,anchor) = "
                             f"{(d2 - d4) / d4:.4f}")
                    L.append(f"  （dNLL(Q4,anchor)={d4:.4f}, dNLL(Q2,anchor)={d2:.4f}）")
        if s8r.get("dtype_deviation"):
            L.append(f"- 偏差声明：{s8r['dtype_deviation']}")
    else:
        L.append("_S8_BF16_ANCHOR 未运行_")
    L.append("")

    # ---- 第三方 ----
    L.append("## 7. 第三方量化（仅探索）\n")
    if sxr:
        L.append(f"- 文件：`{sxr['path']}`，{sxr['file_gb']} GB，{sxr['bpw_effective']} bpw")
        L.append(f"- **不参与公平比较**：{sxr['provenance']['why_not']}")
    else:
        L.append("_SX_EXPLORE 未运行_")
    L.append("")

    # ---- 结论边界 ----
    L.append("## 8. 结论边界\n")
    L.append("1. 本报告所有字节数字来自**文件读取**，不是名义 bpw 乘参数。")
    L.append("2. NLL 的配对 CI 基于分组 bootstrap；**CI 跨 0 表示未测出差异，不是等价**。")
    L.append("3. 单张 T4、8B、单一评测语料，**不能外推到 27B 或 M4 Pro**。")
    L.append("4. T4 无原生 bf16，锚点用 f16 时与论文口径有偏差，已在第 6 节声明。")
    L.append("5. 第三方量化不进比较表；进了就是缺陷（S9 有断言）。")
    L.append("")

    md = "\n".join(L)
    (OUT / "report.md").write_text(md, encoding="utf-8")

    results = {
        "model": CFG["MODEL_ID"],
        "generated_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "stages": {k: {"status": (STATE.get(k) or {}).get("status"),
                       "elapsed_sec": (STATE.get(k) or {}).get("elapsed_sec"),
                       "error": (STATE.get(k) or {}).get("error")}
                   for k in STAGES},
        "config_snapshot": {k: v2 for k, v2 in CFG.items() if k != "STAGE_BUDGET_SEC"},
        "s0_env": env, "s1_smoke": smoke,
        "s3_imatrix": s3r, "s4_quant": s4r, "s5_bytes": s5r,
        "s6_nll": s6r, "s7_quality": s7r, "s8_anchor": s8r, "sx_explore": sxr,
        "disclaimer": ("自建量化与第三方量化的结果不可混用；"
                       "所有 NLL 均为该文件在该语料上的观测，不构成能力结论。"),
    }
    write_json(OUT / "results.json", results)

    print(md)
    print("=" * 78)
    print("产物:")
    print(f"  {OUT/'report.md'}")
    print(f"  {OUT/'results.json'}")
    for p in sorted(OUT.glob("*.json")):
        print(f"  {p}  ({p.stat().st_size} B)")
    return {"report": str(OUT / "report.md"), "results": str(OUT / "results.json"),
            "stage_status": {k: (STATE.get(k) or {}).get("status") for k in STAGES}}


s9 = run_stage("S9_REPORT", stage_s9_report)


In [ ]:
# 最终产物清单 —— 从这里下载，或点 Kaggle 右上角 "Save Version"
print("=" * 78)
for p in sorted(OUT.iterdir()):
    print(f"{p.stat().st_size:>10d} B  {p.name}")
print("-" * 78)
for p in sorted(MDLS.glob("*.gguf")):
    print(f"{p.stat().st_size/2**30:>10.3f} GB  {p.name}")
print("-" * 78)
print(f"断点文件: {STATE_PATH}")
print("阶段状态:")
for k in STAGES:
    r = STATE.get(k) or {}
    print(f"  {k:16s} switch={str(STAGES[k]):5s} status={r.get('status', '-')}"
          f"  {('err: ' + str(r.get('error'))[:70]) if r.get('error') else ''}")
print("=" * 78)
print("提醒：Kaggle 的 /kaggle/working 只有约 20 GB。")
print("      两个 GGUF（2.4 + 4.3 GB）+ imatrix 约 7 GB，余量够；不要在里面再放 base。")


---

## 后续模块规划（本版未实现）

| 模块 | 内容 | 阻塞点 |
|---|---|---|
| **M1 位宽阶梯** | 在同一 base 上再加 `Q4_K_M` / `Q6_K` / `Q8_0`，形成 5 点阶梯，用 `rho` 曲线判断 2.3 bpw 落在哪一段 | 需要更多磁盘与时间，S4 的预算要翻倍 |
| **M2 真 KL** | `llama-perplexity --kl-divergence-base <logits.npy>`，先跑锚点存 logits，再算 KL(Q2‖anchor) | 锚点在 T4 上跑不了（S8 拒绝）；logits 文件对 8B×8k token 约 5 GB |
| **M3 任务级** | GSM8K 子集 / HumanEval+ / BFCL 格式子集 | 需要挂载数据集；8B 在 2.3 bpw 上可能接近地板 |
| **M4 端到端 tok/s** | `llama-bench` 的 `pp512` / `tg128`，报兑现率 `R = (t4/t2)/(B4/B2)` | 需要 CUDA 构建 + 足够长的测量窗口 |
| **M5 跨平台锚点** | 同一 checkpoint 在 A100 40/80 上跑 BF16 锚点，与 T4 的 f16 锚点对齐 | 需要第二张卡；**吞吐不可跨平台迁移，只有格式与质量可以** |

### 关于 M4 的一个提醒

`R = (t_4 / t_2) / (B_4 / B_2)`，理想情形 `R = 1`。**不要写反成 `(B_4/B_2)/(t_2/t_4)`** —— 那样在理想情形会得到 `1.84² = 3.39`，任何结果都会"通过"。这条是本项目踩过的坑。

而且 `R` 只是**带宽受限区间的上界兑现率**。压到极低 bytes/token 之后会转成 latency-bound，实际加速低于字节比。

---

## 已知限制

1. **T4 无原生 bf16（CC 7.5）**。S1 会实测 fp16 vs bf16 的 TFLOP/s 比值；如果 bf16 明显慢，说明走了模拟路径，锚点必须用 f16。
2. **单张 T4 装不下 8B 高精度权重**（f16 = 15.25 GiB vs 约 15.0 GiB 可用，且未计 KV）。S8 默认返回 `status=skipped` 并给出所需/可用的具体数字，而不是硬撑到 OOM。
3. **imatrix 从 Q8_0 base 算**时，与论文常用的 f16 base 有差异。这一项会写进 `S3` 的 provenance。
4. **隔离只能做到内容级**：S6 用前 200 字符做行级重叠检测。如果校准集和评测集来自同一文档的不同段落，这个方法检测不出来 —— 所以必须靠 `CALIB_SOURCE` / `EVAL_SOURCE` 声明不同 split。
5. **`/kaggle/working` 约 20 GB**。base 只能只读挂载，产物放 working。
6. **Kaggle GPU 会话 9 小时**。`MAX_WALL_SEC` 默认 8 小时留了余量。
7. **S7 的样本量极小**（8 prompt × 32 token），只能发现整体崩塌，不能排序细微质量。


---

## 在 Kaggle 上手动执行

1. **新建 Notebook** → `File` → `Import Notebook` → 上传 `trlab_kaggle.ipynb`
2. 右侧 `Settings`：
   - `Accelerator` = **GPU T4 ×2**（Notebook 只用第 0 张；第二张不会自动用上）
   - `Internet` = **Off**（本 Notebook 不需要联网；需要 clone llama.cpp 时才打开并把 `ALLOW_NETWORK=True`）
3. 右侧 `Add Input` → 挂载你的 Dataset（base GGUF / 校准语料 / 评测语料 / 可选 llama.cpp 源码）
4. 跑 `S0_ENV`，从输出里抄出 `/kaggle/input/...` 的真实路径
5. 回到配置 cell，把路径填进 `CFG`，把要跑的阶段改成 `True`
6. **从上到下逐段跑**。每段结束都会把状态写进 `trlab/ckpt/state.json`，中断后重跑会自动跳过已完成的阶段
7. 最后跑 `S9_REPORT`，产物在 `/kaggle/working/trlab/out/`

### 如果只是想先确认环境

只保留 `S0_ENV` / `S1_SMOKE` / `S9_REPORT` 为 `True`，其余全 `False`，直接 `Run All`。约 2–4 分钟，不碰任何大文件。

### 断点恢复怎么工作

```
trlab/ckpt/state.json      # {"S0_ENV": {"status": "ok", ...}, "S3_IMATRIX": {"status": "error", ...}}
trlab/out/<STAGE>.json     # 每个阶段的结果，独立可读
trlab/logs/*.log           # 外部命令的完整输出
```

`status == "ok"` 的阶段会被跳过。想强制重跑，把 `CFG["FORCE_RERUN"] = True`。


---

## 一句话总结

这个 Notebook 在首版只做**三件不花钱的事**：把环境问清楚、把 GGUF 字节解析器验证对、把预算算出来。**任何需要下载、需要大显存、需要长时间运行的东西，都必须你手动打开开关。**

如果 `S0_ENV` 的输出告诉你 `/kaggle/input` 下没有 GGUF，那 S3–S7 一定跑不了 —— 这是设计如此，不是 bug。
